In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Load data
df = pd.read_csv("../data/interim/daily_clean.csv")

# Convert date
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# Returns
df['r_oil'] = np.log(df['Brent futures'] / df['Brent futures'].shift(1))
df['r_sp500'] = np.log(df['S&P500'] / df['S&P500'].shift(1))

# Volatility proxy
df['vol_sp500'] = df['r_sp500']**2

# Lag variables
df['r_oil_lag'] = df['r_oil'].shift(1)
df['r_sp500_lag'] = df['r_sp500'].shift(1)
df['vol_lag'] = df['vol_sp500'].shift(1)

# Dummy early 2000s
df['D_2000'] = ((df['Date'].dt.year >= 1997) & (df['Date'].dt.year <= 2006)).astype(int)

# Interaction
df['interaction'] = df['r_oil'] * df['D_2000']

# Drop NA
df = df.dropna()

print(df.describe())

                      Date  Brent futures       S&P500   MSCI World  \
count                 9441    9441.000000  9441.000000  9441.000000   
mean   2008-02-06 00:00:00      53.532114  1847.939068  1510.438304   
min    1990-01-03 00:00:00       9.750000   295.460000   423.140000   
25%    1999-01-20 00:00:00      21.310000   927.230000   902.870000   
50%    2008-02-06 00:00:00      51.840000  1305.140000  1269.730000   
75%    2017-02-22 00:00:00      76.270000  2341.590000  1805.510000   
max    2026-03-11 00:00:00     146.600000  6978.600000  4578.080000   
std                    NaN      32.225363  1485.451556   883.522751   

       US 10-year Rate  US 2-year Rate  High yield index         Gold  \
count      9441.000000     9441.000000       9441.000000  9441.000000   
mean          4.238533        3.230335          9.098458  1003.411560   
min           0.506900        0.101300          3.530000   252.550000   
25%           2.605200        0.934300          6.840000   367.15000

In [6]:
X = df[['r_oil', 'r_oil_lag', 'r_sp500_lag', 'D_2000', 'interaction']]
X = sm.add_constant(X)

y = df['r_sp500']

model = sm.OLS(y, X).fit()
print(df.describe())

                      Date  Brent futures       S&P500   MSCI World  \
count                 9441    9441.000000  9441.000000  9441.000000   
mean   2008-02-06 00:00:00      53.532114  1847.939068  1510.438304   
min    1990-01-03 00:00:00       9.750000   295.460000   423.140000   
25%    1999-01-20 00:00:00      21.310000   927.230000   902.870000   
50%    2008-02-06 00:00:00      51.840000  1305.140000  1269.730000   
75%    2017-02-22 00:00:00      76.270000  2341.590000  1805.510000   
max    2026-03-11 00:00:00     146.600000  6978.600000  4578.080000   
std                    NaN      32.225363  1485.451556   883.522751   

       US 10-year Rate  US 2-year Rate  High yield index         Gold  \
count      9441.000000     9441.000000       9441.000000  9441.000000   
mean          4.238533        3.230335          9.098458  1003.411560   
min           0.506900        0.101300          3.530000   252.550000   
25%           2.605200        0.934300          6.840000   367.15000

In [7]:
X = df[['r_oil', 'r_oil_lag', 'vol_lag', 'D_2000', 'interaction']]
X = sm.add_constant(X)

y = df['vol_sp500']

model_vol = sm.OLS(y, X).fit()

print(model_vol.summary())

                            OLS Regression Results                            
Dep. Variable:              vol_sp500   R-squared:                       0.105
Model:                            OLS   Adj. R-squared:                  0.105
Method:                 Least Squares   F-statistic:                     222.4
Date:                Sat, 21 Mar 2026   Prob (F-statistic):          4.45e-225
Time:                        18:19:27   Log-Likelihood:                 59766.
No. Observations:                9441   AIC:                        -1.195e+05
Df Residuals:                    9435   BIC:                        -1.195e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const        8.669e-05   5.36e-06     16.186      